# Clean the dataset

Runs the cleaning steps on the raw PhiUSIIL CSV, in order, and prints how many rows each step removes. Every step reads from the raw file, so running the notebook twice gives the same counts.

| Step | Rule |
|---|---|
| 1. Duplicate URLs | A row whose `URL` exactly matches another row's is a duplicate, and the first copy is kept. If copies of a URL have different labels, every copy is removed and counted separately. |
| 2. Missing, empty and out-of-range values | Out of range means a count below 0, a ratio or rate outside 0 to 1, a score outside 0 to 100, a 0/1 column or the label holding anything else. The `Title` placeholder `0` is blanked. Every other problem removes its row. |

The CSV is read so that only truly empty cells count as missing (`keep_default_na=False`). pandas would otherwise turn text such as `NA` or `null` into missing values. The file uses `na` for Namibia's real domain, `.na`, which must stay as text.

The steps live in `ml/cleaning/` and are tested in `ml/tests/`. This notebook only runs them. Deduplication happens here, before any train/test split.

Get the CSV first by following `ml/README.md`. Links in the data are only ever handled as text.

In [1]:
import sys
from pathlib import Path

import pandas as pd

CSV_NAME = "PhiUSIIL_Phishing_URL_Dataset.csv"


def find_ml_dir(start: Path) -> Path:
    """Walk up to the repo root (the folder holding AGENTS.md), so the notebook runs from any working directory inside the repo."""
    for folder in [start, *start.parents]:
        if (folder / "AGENTS.md").exists() and (folder / "ml").is_dir():
            return folder / "ml"
    raise FileNotFoundError("Could not find the ml/ folder. Open this notebook from inside the LinkGuard repo.")


ML_DIR = find_ml_dir(Path.cwd().resolve())
DATA_PATH = ML_DIR / "data" / CSV_NAME
sys.path.insert(0, str(ML_DIR))

from cleaning.duplicates import remove_duplicate_urls  # noqa: E402
from cleaning.values import clean_values  # noqa: E402

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}.\n"
        "Download it by following 'Get the dataset' in ml/README.md, then run this notebook again."
    )

raw = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[""])
print(f"Raw file: {len(raw):,} rows")

Raw file: 235,795 rows


## Step 1: duplicate URLs

In [2]:
duplicates = remove_duplicate_urls(raw)

print(f"Rows before:                    {duplicates.rows_before:,}")
print(f"Extra copies removed:           {duplicates.duplicate_rows_removed:,}")
print(f"URLs with both labels:          {duplicates.conflicting_urls:,}")
print(f"  rows removed for that reason: {duplicates.conflicting_rows_removed:,}")
print(f"Rows after:                     {duplicates.rows_after:,}")

Rows before:                    235,795
Extra copies removed:           425
URLs with both labels:          0
  rows removed for that reason: 0
Rows after:                     235,370


## Why this is not 808

Other analyses report about 808 duplicates. The difference comes from what counts as a duplicate:

- **808** is the number of rows whose **numeric feature values** exactly match an earlier row's, even though their URLs are different.
- **This step** follows the story's spec: a duplicate is a row with the **same `URL`**.

The cell below works out both numbers from the raw file.

In [3]:
same_url = raw.duplicated("URL").sum()
same_numeric_features = raw.select_dtypes("number").duplicated().sum()

print(f"Extra rows with the same URL:                 {same_url:,}")
print(f"Extra rows with the same numeric features:    {same_numeric_features:,}")

Extra rows with the same URL:                 425
Extra rows with the same numeric features:    808


## What the duplicates look like

The cell below shows how many copies each duplicated URL has, which label they carry, and how often the two copies' features disagree.

In [4]:
copies = raw[raw.duplicated("URL", keep=False)]
copies_per_url = copies.groupby("URL").size()

print(f"Duplicated URLs: {copies_per_url.size:,}")
print(f"Copies per duplicated URL: {copies_per_url.value_counts().sort_index().to_dict()}  (copies: number of URLs)")
print(f"Raw label of their copies: {copies['label'].value_counts().to_dict()}  (raw file: 0 = phishing)")

feature_columns = [column for column in raw.columns if column not in ("FILENAME", "URL", "label")]
differs = copies.groupby("URL")[feature_columns].nunique().gt(1)
print(f"URLs whose two copies differ in at least one feature: {differs.any(axis=1).sum():,} of {copies_per_url.size:,}")
print(f"URLs whose copies even differ in URLLength:           {differs['URLLength'].sum():,}")

Duplicated URLs: 425
Copies per duplicated URL: {2: 425}  (copies: number of URLs)
Raw label of their copies: {0: 850}  (raw file: 0 = phishing)
URLs whose two copies differ in at least one feature: 425 of 425
URLs whose copies even differ in URLLength:           425


Every duplicated URL appears exactly twice, and both copies are labelled phishing. The two copies are never identical rows. Every pair differs in its page features, and even in `URLLength`, although the URL text is exactly the same. So the feature values weren't worked out from the URL text alone, and each copy is probably a separate crawl of the same link.

The spec's rule is to keep the first copy. Both copies share a label, so the choice doesn't change the label balance. It does mean the kept row's features come from one crawl only.

## Step 2: missing, empty and out-of-range values

Runs on the output of step 1. The first table counts problems in every column. The second lists only the columns that have a problem, with the action taken.

In [5]:
values = clean_values(duplicates.cleaned)

problems = values.problems
print(f"Checked {len(problems)} columns. Problems per column:\n")
print(problems.to_string())

Checked 56 columns. Problems per column:

                            missing  empty  out_of_range
FILENAME                          0      0             0
URL                               0      0             0
URLLength                         0      0             0
Domain                            0      0             0
DomainLength                      0      0             0
IsDomainIP                        0      0             0
TLD                               0      0             0
URLSimilarityIndex                0      0             0
CharContinuationRate              0      0             0
TLDLegitimateProb                 0      0             0
URLCharProb                       0      0             0
TLDLength                         0      0             0
NoOfSubDomain                     0      0             0
HasObfuscation                    0      0             0
NoOfObfuscatedChar                0      0             0
ObfuscationRatio                  0      0    

In [6]:
ACTIONS = {
    ("Title", "empty"): "The text '0' marks a page with no title (HasTitle = 0). Fixed: the title is blanked and the row kept.",
}
DEFAULT_ACTION = "Row removed: the model can't use a row with a broken feature or label."

found = problems.stack()
found = found[found > 0]
actions = pd.DataFrame(
    [
        {"Column": column, "Problem": kind, "Rows": int(count), "Action": ACTIONS.get((column, kind), DEFAULT_ACTION)}
        for (column, kind), count in found.items()
    ],
    columns=["Column", "Problem", "Rows", "Action"],
)
print(actions.to_string(index=False) if len(actions) else "No problems found.")
print(f"\nTitles fixed: {values.titles_fixed:,}")
print(f"Rows removed: {values.rows_removed:,}")
print(f"Rows after:   {values.rows_after:,}")

Column Problem  Rows                                                                                                Action
 Title   empty 32612 The text '0' marks a page with no title (HasTitle = 0). Fixed: the title is blanked and the row kept.

Titles fixed: 32,612
Rows removed: 0
Rows after:   235,370


By the story's rules the file is in good shape. There are no missing cells, no negative counts, every ratio and rate is within 0 to 1, every score is within 0 to 100, and every 0/1 column and the label hold only 0 or 1. The one problem is the `Title` placeholder. Pages with no title store the text `0` as their title, which a text feature would treat as a real word. It is blanked, so `HasTitle = 0` and an empty title now agree.

A few pages that do have a title (`HasTitle = 1`) are literally titled `0`. Those are kept as they are.

## Row counts by step

Later cleaning steps add a row to this table.

In [7]:
steps = pd.DataFrame(
    [
        {"Step": "Raw file", "Rows removed": 0, "Rows left": len(raw)},
        {
            "Step": "1. Duplicate URLs",
            "Rows removed": duplicates.rows_before - duplicates.rows_after,
            "Rows left": duplicates.rows_after,
        },
        {
            "Step": "2. Missing, empty and out-of-range values",
            "Rows removed": values.rows_removed,
            "Rows left": values.rows_after,
        },
    ]
)
print(steps.to_string(index=False))

                                     Step  Rows removed  Rows left
                                 Raw file             0     235795
                        1. Duplicate URLs           425     235370
2. Missing, empty and out-of-range values             0     235370
